# 데이터 전처리 파이프라인\n이 노트북은 프로젝트에 사용되는 모든 데이터를 전처리하는 과정을 보여줍니다.\n각 단계의 복잡한 로직은 `src/data_processing.py`와 `src/config.py`에 모듈화되어 있습니다.

## 1. 개인정보보호법 판례 데이터 처리

In [ ]:
import sys\nimport os\n# 프로젝트 루트를 경로에 추가\nsys.path.append(os.path.abspath('..'))\n\nfrom src import config\nfrom src.data_processing import process_tiff_images_to_csv, apply_semantic_chunking_to_csv\n\n# 1. TIFF 이미지에서 텍스트 추출 (OCR)\nprocess_tiff_images_to_csv(\n    image_folder=config.TIFF_IMAGE_DIR,\n    output_csv=config.OCR_OUTPUT_CSV\n)\n\n# 2. 추출된 텍스트에 Semantic Chunking 적용\napply_semantic_chunking_to_csv(\n    input_csv=config.OCR_OUTPUT_CSV,\n    output_csv=config.LEGAL_SEMANTIC_CHUNK_OUTPUT_CSV\n)

## 2. 법률 PDF 데이터 처리

In [ ]:
from src import config\nfrom src.data_processing import LawPDFProcessor\n\n# 법률 PDF 처리 파이프라인 실행\npdf_processor = LawPDFProcessor(\n    pdf_dir=config.PDF_DIR,\n    output_csv=config.LAW_CHUNK_OUTPUT_CSV,\n    max_chars=config.LAW_CHUNK_MAX_CHARS,\n    overlap=config.LAW_CHUNK_OVERLAP,\n    min_chars=config.LAW_CHUNK_MIN_CHARS,\n    dedup=config.DEDUP_HASHES\n)\npdf_processor.run_pipeline()

## 3. 위키백과 데이터 처리

In [ ]:
from src import config\nfrom src.data_processing import WikipediaProcessor\n\nwiki_processor = WikipediaProcessor(\n    output_dir=config.PROCESSED_DATA_DIR\n)\n\n# 1. 보안 관련 용어에 해당하는 위키피디아 문서 가져오기\nwiki_processor.fetch_articles_by_title(\n    output_csv=config.WIKI_SECURITY_TITLES_CSV,\n    missing_log_file=config.MISSING_WIKI_TITLES_LOG\n)\n\n# 2. 가져온 문서들에 대해 Chunking 수행\nwiki_processor.run_chunking_pipeline(\n    input_csv=config.WIKI_SECURITY_TITLES_CSV,\n    output_csv=config.WIKI_CHUNKS_OUTPUT_CSV\n)